In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

# Harness-only conversion helpers: the mined fixture is stored in target-side
# form, while the oracle must receive the semantically equivalent pandas form.
def _to_pandas_fixture(value):
    if isinstance(value, pl.DataFrame):
        return value.to_pandas()
    if isinstance(value, pl.Series):
        return value.to_pandas()
    if isinstance(value, list):
        return [_to_pandas_fixture(item) for item in value]
    if isinstance(value, tuple):
        return tuple(_to_pandas_fixture(item) for item in value)
    if isinstance(value, dict):
        return {key: _to_pandas_fixture(item) for key, item in value.items()}
    if isinstance(value, SimpleNamespace):
        return SimpleNamespace(**{
            key: _to_pandas_fixture(item) for key, item in vars(value).items()
        })
    return value

def _to_polars_fixture(value):
    if isinstance(value, pd.DataFrame):
        return pl.from_pandas(value)
    if isinstance(value, pd.Series):
        return pl.from_pandas(value)
    if isinstance(value, list):
        return [_to_polars_fixture(item) for item in value]
    if isinstance(value, tuple):
        return tuple(_to_polars_fixture(item) for item in value)
    if isinstance(value, dict):
        return {key: _to_polars_fixture(item) for key, item in value.items()}
    if isinstance(value, SimpleNamespace):
        return SimpleNamespace(**{
            key: _to_polars_fixture(item) for key, item in vars(value).items()
        })
    return value


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- lines_filter_words ---
FIX_LINES_FILTER_WORDS_OCR_DF = SimpleNamespace(df=pl.DataFrame({"class": ["ocrx_word", "ocrx_word", "ocrx_block"], "confidence": [90, None, 10], "value": ["a", "b", "block"], "x1": [0, 10, 0], "y1": [0, 0, 20], "x2": [5, 15, 10], "y2": [5, 5, 30]}))

# --- lines_from_dicts_lazy ---
FIX_LINES_FROM_DICTS_LAZY_LINES = [SimpleNamespace(dict={"x1":0,"x2":300,"y1":15,"y2":15,"width":300,"height":1,"length":300,"vertical":False,"line_id":0}), SimpleNamespace(dict={"x1":100,"x2":100,"y1":0,"y2":200,"width":1,"height":200,"length":200,"vertical":True,"line_id":1})]

# --- lines_groupby_filter ---
FIX_LINES_GROUPBY_FILTER_DF_W_L = pl.DataFrame({"x1":[10,110],"y1":[10,10],"x2":[100,200],"y2":[30,30],"x1_line":[0,0],"x2_line":[300,300],"y1_line":[15,15],"y2_line":[15,15],"width":[90,90],"height":[20,20],"vertical":[False,False],"intersection":[60,70],"line_id":[0,1],"length":[300,300]})

print("✅ Fixtures loaded")
OCRDataframe = SimpleNamespace  # mock for testing


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_lines_filter_words(ocr_df):
    df_words = ocr_df.df[ocr_df.df['class'] == 'ocrx_word']
    df_words = df_words[(df_words['confidence'] >= 50) | df_words['confidence'].isna()]
    return df_words

def before_lines_from_dicts_lazy(lines):
    df_lines = pd.DataFrame(data=[line.dict for line in lines])
    df_lines['length'] = pd.concat([df_lines['width'], df_lines['height']], axis=1).max(axis=1)
    df_lines['vertical'] = (df_lines['x1'] == df_lines['x2'])
    df_lines['line_id'] = range(len(df_lines))
    df_lines.columns = ['x1_line', 'x2_line', 'y1_line', 'y2_line', 'width', 'height', 'length', 'vertical', 'line_id']
    return df_lines

def before_lines_groupby_filter(df_w_l):
    df_inter = (df_w_l.groupby(['line_id', 'length'])
                .agg(intersection=('intersection', np.sum))
                .reset_index())
    intersecting_lines = df_inter[df_inter['intersection'] / df_inter['length'] > 0.5]['line_id'].values.tolist()
    return intersecting_lines

_oracle_lines_filter_words = before_lines_filter_words
def before_lines_filter_words(*args, **kwargs):
    return _oracle_lines_filter_words(
        *[_to_pandas_fixture(value) for value in args],
        **{key: _to_pandas_fixture(value) for key, value in kwargs.items()},
    )


In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_lines_filter_words(ocr_df):
    pd = pl  # LLM used `import polars as pd`
    df_words = ocr_df.df.filter(pd.col('class') == 'ocrx_word')
    df_words = df_words.filter((pd.col('confidence') >= 50) | pd.col('confidence').is_null())
    return df_words

def gen_lines_from_dicts_lazy(lines):
    df_lines = pl.DataFrame(data=[line.dict for line in lines])
    df_lines = df_lines.with_columns(
        pl.max_horizontal(["width", "height"]).alias("length"),
        (pl.col("x1") == pl.col("x2")).alias("vertical"),
        pl.arange(0, pl.len()).alias("line_id"),
    )
    df_lines = df_lines.select([
        pl.col("x1").alias("x1_line"),
        pl.col("x2").alias("x2_line"),
        pl.col("y1").alias("y1_line"),
        pl.col("y2").alias("y2_line"),
        "width",
        "height",
        "length",
        "vertical",
        "line_id",
    ])
    return df_lines

def gen_lines_groupby_filter(df_w_l):
    df_inter = (df_w_l.group_by(['line_id', 'length'])
                .agg(pl.col('intersection').sum().alias('intersection')))
    intersecting_lines = df_inter.filter(pl.col('intersection') / pl.col('length') > 0.5)['line_id'].to_list()
    return intersecting_lines

# ── Test harness type adapters ─────────────────────────────────────────────
def _to_pandas_fixture(obj):
    if isinstance(obj, pl.Series):
        return obj.to_pandas()
    if isinstance(obj, pl.DataFrame):
        return obj.to_pandas()
    if isinstance(obj, pl.LazyFrame):
        return obj.collect().to_pandas()
    if isinstance(obj, list):
        return [_to_pandas_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_pandas_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_pandas_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), (pl.DataFrame, pl.LazyFrame)):
        return SimpleNamespace(df=_to_pandas_fixture(obj.df))
    return obj

def _to_polars_fixture(obj):
    if isinstance(obj, pd.Series):
        return pl.Series(obj.name or "series", obj.to_list())
    if isinstance(obj, pd.DataFrame):
        return pl.from_pandas(obj)
    if isinstance(obj, list):
        return [_to_polars_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_polars_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_polars_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), pd.DataFrame):
        return SimpleNamespace(df=_to_polars_fixture(obj.df))
    return obj

def _wrap_before_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_pandas_fixture(self.df)
        try:
            return fn(*[_to_pandas_fixture(a) for a in args], **{k: _to_pandas_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

def _wrap_gen_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_polars_fixture(self.df)
        try:
            return fn(*[_to_polars_fixture(a) for a in args], **{k: _to_polars_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

for _name, _fn in list(globals().items()):
    if callable(_fn) and _name.startswith("before_"):
        globals()[_name] = _wrap_before_func(_fn)
    elif callable(_fn) and _name.startswith("gen_"):
        globals()[_name] = _wrap_gen_func(_fn)


In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if hasattr(r, "df"):
        r = r.df
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: lines_filter_words ===

# L1 smoke – generated
try:
    _r = gen_lines_filter_words(FIX_LINES_FILTER_WORDS_OCR_DF)
    print("✅ L1 smoke gen_lines_filter_words: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_lines_filter_words: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_lines_filter_words(FIX_LINES_FILTER_WORDS_OCR_DF)
    print("✅ L1 smoke before_lines_filter_words: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_lines_filter_words: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_lines_filter_words(FIX_LINES_FILTER_WORDS_OCR_DF)
    _rg = gen_lines_filter_words(FIX_LINES_FILTER_WORDS_OCR_DF)
    compare(_rb, _rg, "lines_filter_words")
except Exception as _e:
    print(f"❌ L2 equivalence lines_filter_words: setup error — {type(_e).__name__}: {_e}")

# L3 edge - schema-bearing empty OCR data.
try:
    _empty = SimpleNamespace(df=pl.DataFrame(schema={
        "class": pl.String, "confidence": pl.Float64, "value": pl.String,
        "x1": pl.Int64, "y1": pl.Int64, "x2": pl.Int64, "y2": pl.Int64,
    }))
    _rb = before_lines_filter_words(_empty)
    _rg = gen_lines_filter_words(_empty)
    compare(_rb, _rg, "L3 edge lines_filter_words empty OCR", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge lines_filter_words empty OCR: {type(_e).__name__}: {_e}")
